In [1]:
# CÉLULA 1: Configuração + Schema Padrão
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
import json
import pandas as pd
import hashlib
import zipfile
from urllib.request import urlretrieve
import socket
import time
import sys

# Configurações
BASE_URL = "https://portaldatransparencia.gov.br/download-de-dados/bpc/{ano_mes}"
BRONZE_PATH = "/lakehouse/default/Files/bronze_bpc"
SILVER_PATH = "/lakehouse/default/Files/silver_bpc"
METADATA_PATH = "/lakehouse/default/Files/metadata/bpc"

ANO_INICIO = 2019
MES_INICIO = 1

TIMEOUT_SECONDS = 300
MAX_RETRIES = 3
RETRY_DELAY = 10

# SCHEMA PADRÃO - Define tipos fixos para todas as colunas
SCHEMA_BPC = {
    'MÊS COMPETÊNCIA': 'string',
    'MÊS REFERÊNCIA': 'string',
    'UF': 'string',
    'CÓDIGO MUNICÍPIO SIAFI': 'string',  # STRING evita conflito BIGINT/DOUBLE
    'NOME MUNICÍPIO': 'string',
    'NIS BENEFICIÁRIO': 'string',
    'CPF BENEFICIÁRIO': 'string',
    'NOME BENEFICIÁRIO': 'string',
    'NIS REPRESENTANTE LEGAL': 'string',  # STRING evita conflito
    'CPF REPRESENTANTE LEGAL': 'string',
    'NOME REPRESENTANTE LEGAL': 'string',
    'NÚMERO BENEFÍCIO': 'string',  # STRING evita conflito
    'BENEFÍCIO CONCEDIDO JUDICIALMENTE': 'string',
    'VALOR PARCELA': 'string',  # Vai precisar tratar depois (vírgula decimal)
}

def get_ultimo_periodo_disponivel():
    """Retorna YYYYMM do mês anterior"""
    hoje = datetime.now()
    mes = hoje.month - 1
    ano = hoje.year
    if mes == 0:
        mes = 12
        ano -= 1
    return f"{ano}{mes:02d}"

StatementMeta(, 075349df-5815-4039-a572-2d43ffe2d993, 3, Finished, Available, Finished, False)

In [2]:
# CÉLULA 2: Funções de Metadados
def inicializar_metadata(metadata_path):
    """Cria estrutura de metadados se não existir"""
    Path(metadata_path).mkdir(parents=True, exist_ok=True)
    
    controle_file = Path(metadata_path) / "controle_carga.csv"
    if not controle_file.exists():
        df = pd.DataFrame(columns=[
            'ano_mes', 'data_carga', 'status', 
            'registros', 'tamanho_mb', 'hash_arquivo'
        ])
        df.to_csv(controle_file, index=False)

def registrar_carga(metadata_path, ano_mes, status, registros=0, 
                   tamanho_mb=0, hash_arquivo=None):
    """Registra metadados de uma carga"""
    controle_file = Path(metadata_path) / "controle_carga.csv"
    df = pd.read_csv(controle_file)
    
    novo_registro = pd.DataFrame([{
        'ano_mes': ano_mes,
        'data_carga': datetime.now().isoformat(),
        'status': status,
        'registros': registros,
        'tamanho_mb': tamanho_mb,
        'hash_arquivo': hash_arquivo
    }])
    
    df = df[df['ano_mes'] != ano_mes]
    df = pd.concat([df, novo_registro], ignore_index=True)
    df.to_csv(controle_file, index=False)

def get_periodos_carregados(metadata_path):
    """Retorna lista de períodos já carregados com sucesso"""
    controle_file = Path(metadata_path) / "controle_carga.csv"
    if not controle_file.exists():
        return []
    df = pd.read_csv(controle_file)
    return df[df['status'] == 'sucesso']['ano_mes'].tolist()

def atualizar_status_geral(metadata_path, info):
    """Atualiza status geral da última execução"""
    status_file = Path(metadata_path) / "ultima_atualizacao.json"
    with open(status_file, 'w') as f:
        json.dump({
            **info,
            'data_atualizacao': datetime.now().isoformat()
        }, f, indent=2)

def get_status_geral(metadata_path):
    """Retorna status geral da última execução"""
    status_file = Path(metadata_path) / "ultima_atualizacao.json"
    if not status_file.exists():
        return None
    with open(status_file, 'r') as f:
        return json.load(f)

StatementMeta(, 075349df-5815-4039-a572-2d43ffe2d993, 4, Finished, Available, Finished, False)

In [3]:
# CÉLULA 3: Funções de Processamento COM VALIDAÇÃO DE SCHEMA
def calcular_hash(filepath):
    """Calcula hash SHA256 do arquivo"""
    sha256 = hashlib.sha256()
    with open(filepath, 'rb') as f:
        for bloco in iter(lambda: f.read(4096), b''):
            sha256.update(bloco)
    return sha256.hexdigest()

def download_periodo(ano_mes, output_dir):
    """Download de um período específico"""
    url = BASE_URL.format(ano_mes=ano_mes)
    dest = Path(output_dir) / f"bpc_{ano_mes}.zip"
    
    if dest.exists():
        print(f"Arquivo já existe: {dest.name}")
        return True, dest
    
    original_timeout = socket.getdefaulttimeout()
    socket.setdefaulttimeout(TIMEOUT_SECONDS)
    
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(f"Baixando {ano_mes}... (tentativa {attempt}/{MAX_RETRIES})")
            urlretrieve(url, dest)
            socket.setdefaulttimeout(original_timeout)
            print(f"✓ {dest.name} baixado")
            return True, dest
        except Exception as exc:
            print(f"✗ Tentativa {attempt} falhou: {exc}", file=sys.stderr)
            if dest.exists():
                dest.unlink()
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY)
    
    socket.setdefaulttimeout(original_timeout)
    return False, None

def processar_zip(zip_path):
    """Extrai e processa CSV do ZIP COM VALIDAÇÃO DE SCHEMA"""
    
    with zipfile.ZipFile(zip_path, 'r') as zf:
        csv_files = [f for f in zf.namelist() if f.endswith('.csv')]
        if not csv_files:
            raise ValueError(f"Nenhum CSV encontrado em {zip_path.name}")
        
        # Ler CSV
        with zf.open(csv_files[0]) as f:
            df = pd.read_csv(f, encoding='latin1', sep=';', low_memory=False)
    
    # APLICAR SCHEMA PADRÃO
    print(f"  📋 Colunas originais: {len(df.columns)}")
    df = aplicar_schema_padrao(df)
    print(f"  ✓ Schema padronizado aplicado")
    
    return df

def aplicar_schema_padrao(df):
    """
    Aplica o schema padrão ao DataFrame:
    - Garante que todas as colunas existam
    - Converte tipos conforme SCHEMA_BPC
    - Adiciona colunas faltantes como NULL
    """
    df_padronizado = df.copy()
    
    # 1. Converter tipos de todas as colunas do schema
    for col, tipo in SCHEMA_BPC.items():
        if col in df_padronizado.columns:
            if tipo == 'string':
                # Converte para string, substituindo NaN por None
                df_padronizado[col] = df_padronizado[col].astype(str)
                df_padronizado[col] = df_padronizado[col].replace('nan', None)
                df_padronizado[col] = df_padronizado[col].replace('', None)
            elif tipo == 'int64':
                df_padronizado[col] = pd.to_numeric(df_padronizado[col], errors='coerce').astype('Int64')
            elif tipo == 'float64':
                df_padronizado[col] = pd.to_numeric(df_padronizado[col], errors='coerce')
        else:
            # Se a coluna não existe, cria com NULL
            print(f"  ⚠️ Coluna '{col}' não encontrada, criando com NULL")
            df_padronizado[col] = None
    
    # 2. Remover colunas que não estão no schema (se houver extras)
    colunas_extras = set(df_padronizado.columns) - set(SCHEMA_BPC.keys())
    if colunas_extras:
        print(f"  ⚠️ Removendo colunas extras: {colunas_extras}")
        df_padronizado = df_padronizado.drop(columns=list(colunas_extras))
    
    # 3. Reordenar colunas na ordem do schema
    df_padronizado = df_padronizado[list(SCHEMA_BPC.keys())]
    
    return df_padronizado

def identificar_periodos_faltantes(metadata_path):
    """Identifica períodos que precisam ser carregados"""
    periodos_carregados = set(get_periodos_carregados(metadata_path))
    ultimo_disponivel = get_ultimo_periodo_disponivel()
    
    todos_periodos = []
    ano = ANO_INICIO
    mes = MES_INICIO
    
    while True:
        periodo = f"{ano}{mes:02d}"
        if periodo > ultimo_disponivel:
            break
        todos_periodos.append(periodo)
        mes += 1
        if mes > 12:
            mes = 1
            ano += 1
    
    faltantes = [p for p in todos_periodos if p not in periodos_carregados]
    return sorted(faltantes)

StatementMeta(, 075349df-5815-4039-a572-2d43ffe2d993, 5, Finished, Available, Finished, False)

In [4]:
# CÉLULA 4: Execução Principal COM VALIDAÇÃO
def executar_carga_incremental():
    """Execução principal - carga incremental com schema validado"""
    
    inicializar_metadata(METADATA_PATH)
    Path(BRONZE_PATH).mkdir(parents=True, exist_ok=True)
    
    periodos_faltantes = identificar_periodos_faltantes(METADATA_PATH)
    
    if not periodos_faltantes:
        print("✓ Todos os períodos já estão carregados!")
        atualizar_status_geral(METADATA_PATH, {
            'status': 'atualizado',
            'mensagem': 'Nenhum novo período para carregar'
        })
        return
    
    print(f"\n📥 Períodos a carregar: {len(periodos_faltantes)}")
    print(f"De {periodos_faltantes[0]} até {periodos_faltantes[-1]}\n")
    
    sucessos, falhas = 0, 0
    
    for ano_mes in periodos_faltantes:
        print(f"\n{'='*60}")
        print(f"Processando: {ano_mes}")
        print(f"{'='*60}")
        
        # 1. Download
        sucesso, zip_path = download_periodo(ano_mes, BRONZE_PATH)
        if not sucesso:
            registrar_carga(METADATA_PATH, ano_mes, 'falha_download')
            falhas += 1
            continue
        
        try:
            # 2. Calcular hash
            hash_arquivo = calcular_hash(zip_path)
            tamanho_mb = zip_path.stat().st_size / (1024 * 1024)
            
            # 3. Processar COM VALIDAÇÃO DE SCHEMA
            df = processar_zip(zip_path)
            num_registros = len(df)
            
            # 4. Adicionar colunas de partição
            ano = ano_mes[:4]
            mes = ano_mes[4:6]
            df['ano_referencia'] = ano
            df['mes_referencia'] = mes
            
            # 5. Salvar na camada Silver
            output_path = Path(SILVER_PATH) / f"ano={ano}" / f"mes={mes}"
            output_path.mkdir(parents=True, exist_ok=True)
            
            # IMPORTANTE: Salvar com PyArrow para garantir schema consistente
            df.to_parquet(
                output_path / "data.parquet",
                engine='pyarrow',
                compression='snappy',
                index=False,
                # PyArrow vai respeitar os tipos do DataFrame pandas
            )
            
            # 6. Registrar metadata
            registrar_carga(
                METADATA_PATH,
                ano_mes=ano_mes,
                status='sucesso',
                registros=num_registros,
                tamanho_mb=tamanho_mb,
                hash_arquivo=hash_arquivo
            )
            
            sucessos += 1
            print(f"✓ {ano_mes}: {num_registros:,} registros processados")
            
        except Exception as e:
            print(f"✗ Erro ao processar {ano_mes}: {e}", file=sys.stderr)
            import traceback
            traceback.print_exc()
            registrar_carga(METADATA_PATH, ano_mes, f'falha_processamento: {str(e)}')
            falhas += 1
    
    # Resumo final
    print(f"\n{'='*60}")
    print(f"RESUMO FINAL")
    print(f"{'='*60}")
    print(f"✓ Sucessos: {sucessos}")
    print(f"✗ Falhas: {falhas}")
    
    atualizar_status_geral(METADATA_PATH, {
        'status': 'completo',
        'periodos_processados': sucessos,
        'falhas': falhas,
        'ultimo_periodo': periodos_faltantes[-1] if periodos_faltantes else None
    })

# EXECUTAR
executar_carga_incremental()

StatementMeta(, 075349df-5815-4039-a572-2d43ffe2d993, 6, Submitted, Running, Running, True)


📥 Períodos a carregar: 85
De 201901 até 202601


Processando: 201901
Baixando 201901... (tentativa 1/3)
✓ bpc_201901.zip baixado
  📋 Colunas originais: 14
  ✓ Schema padronizado aplicado
✓ 201901: 4,631,027 registros processados

Processando: 201902
Baixando 201902... (tentativa 1/3)
✓ bpc_201902.zip baixado


In [ ]:
# CÉLULA 5: Visualizar Status
# Ver o que já foi carregado
status = get_status_geral(METADATA_PATH)
print("📊 Status Geral:")
print(json.dumps(status, indent=2))

print("\n📋 Histórico de Cargas:")
controle = pd.read_csv(Path(METADATA_PATH) / "controle_carga.csv")
display(controle.tail(10))

StatementMeta(, , -1, Waiting, , Waiting, True)